# True JEPA: Phase 1 data collection and training

This notebook runs the isolated pixel-only `true_jepa` experiment. Phase 1 collects random image/action/reward sequences without optimizer updates. Later cells train the JEPA and reward heads from that replay, then run locked post-episode evaluation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!git clone https://github.com/sahiltolani30/learn-a-world.git /content/learn-a-world 2>/dev/null || (cd /content/learn-a-world && git pull origin main)
%cd /content/learn-a-world
!pip install -q -r requirements.txt

In [ ]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
!python -m pytest -q tests/test_true_jepa.py

## Phase 1 — collect a clean random dataset

This uses eight independent environments and uniformly random actions. It performs no JEPA or policy optimizer updates and uses the separate `phase1_random` Drive folder. The command can be repeated to append more episodes.

In [ ]:
%cd /content/learn-a-world
!python -m true_jepa.colab_run \
  --drive-root /content/drive/MyDrive/learn-a-world/phase1_random \
  --episodes 256 \
  --updates 0 \
  --num-envs 8 \
  --max-steps 500 \
  --collection-policy random \
  --collect-only \
  --device cpu

In [ ]:
from pathlib import Path
import numpy as np
import crafter
from true_jepa.replay import PixelReplay

replay_path = Path('/content/drive/MyDrive/learn-a-world/phase1_random/true_jepa/shards/replay.npz')
replay = PixelReplay.load(str(replay_path))
actions = np.concatenate([episode.actions for episode in replay.records])
probe = crafter.Env(seed=0)
action_count = int(probe.action_space.n)
probe.close()
counts = np.bincount(actions, minlength=action_count)
print({'episodes': replay.episodes, 'frames': len(replay), 'action_counts': counts.tolist(), 'actions_seen': int((counts > 0).sum()), 'action_count': action_count})
assert replay.episodes >= 128
assert len(replay) >= 30000
assert int((counts > 0).sum()) == action_count

## Phases 2–3 — train JEPA and reward prediction

Run this only after the Phase 1 audit passes. It trains from the saved replay without collecting new episodes.

In [ ]:
%cd /content/learn-a-world
!python -m true_jepa.colab_run \
  --drive-root /content/drive/MyDrive/learn-a-world/phase1_random \
  --episodes 0 \
  --updates 64 \
  --num-envs 8 \
  --max-steps 500 \
  --train-only \
  --device cuda

## Locked post-episode evaluation

Achievement information is used only after each episode for scoring. It is never passed to the agent's action function.

In [ ]:
%cd /content/learn-a-world
!python -m true_jepa.evaluate \
  --checkpoint /content/drive/MyDrive/learn-a-world/phase1_random/true_jepa/checkpoints/state.pt \
  --episodes 16 \
  --max-steps 500 \
  --seed-start 2050 \
  --device cuda \
  --out /content/drive/MyDrive/learn-a-world/phase1_random/true_jepa/results/eval_phase1_training.json